# 🏎️ تحليل بيانات Formula 1 — المرحلة 3
**الربط (Relational Joins) + هندسة الخصائص (Feature Engineering)**

المدخل: النسخة النظيفة من المرحلة 2 (`/content/f1_clean`).  
المخرجات (`/content/f1_model`):

| الجدول | الـ Grain (صف = ؟) | الغرض |
|---|---|---|
| `fact_results` | نتيجة سائق في سباق (`resultId`) | الجدول المحوري: فوز/منصة/انسحاب/مكاسب المراكز |
| `fact_driver_race_perf` | `(raceId, driverId)` | إحصاءات اللفات والـ Pit Stops (بدون تضخيم الأرقام) |
| `fact_race_summary` | سباق (`raceId`) | نسبة الإكمال، الفائز، الـ Pole، هل فاز صاحب الـ Pole |
| `agg_driver_season` / `agg_constructor_season` | سائق/فريق × موسم | أداء الموسم + البطل |
| `agg_circuit` | حلبة | الانسحاب، الإثارة، استراتيجيات الـ Pit |

## 3.1 تحميل النسخة النظيفة

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50, "display.width", 200, "display.float_format", "{:,.3f}".format)
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE      = Path("/content") if IN_COLAB else Path(os.environ.get("F1_WORKDIR", "/tmp/f1"))
CLEAN_DIR = BASE / "f1_clean"
MODEL_DIR = BASE / "f1_model"; MODEL_DIR.mkdir(parents=True, exist_ok=True)

# لو بدأت جلسة Colab جديدة ومجلد f1_clean غير موجود: ارفع f1_clean.zip (أو شغّل notebook المرحلة 1-2 أولاً)
if not CLEAN_DIR.exists() and IN_COLAB:
    import zipfile
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z: z.extractall(CLEAN_DIR)

NAMES = ["circuits","constructors","drivers","seasons","status","races","results","sprint_results",
         "qualifying","lap_times","pit_stops","practice_sessions","driver_standings",
         "constructor_standings","constructor_results"]
def read_clean(n):
    p = CLEAN_DIR / f"{n}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(CLEAN_DIR / f"{n}.csv")
T = {n: read_clean(n) for n in NAMES}
print({n: len(d) for n, d in T.items()})

## 3.2 خريطة الربط (ERD / Schema)
- **أبعاد (Dimensions):** `drivers`, `constructors`, `circuits`, `status`, `seasons`, و`races` (بُعد الزمن/السباق).  
- **الحقائق (Facts):** `results` هو المحور (grain = نتيجة). باقي الجداول التفصيلية ترتبط بـ `raceId` + `driverId` (أو `constructorId`).
- ⚠️ **مفتاح الربط الحرِج:** `(raceId, driverId)` فريد في كل الجداول التفصيلية **عدا `results`** (قيادة مشتركة في الخمسينات)، لذلك لا نضم `lap_times`/`pit_stops` إلى `results` مباشرة (سيضاعف الأرقام) بل نُجمّعها أولاً في `fact_driver_race_perf`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# (x, y): مواضع الجداول — الأبعاد بالأطراف والحقائق بالمنتصف
POS = {
 "seasons": (0.08, 0.90), "circuits": (0.08, 0.62), "races": (0.36, 0.76),
 "drivers": (0.36, 0.10), "constructors": (0.64, 0.10), "status": (0.92, 0.62),
 "results": (0.64, 0.50), "sprint_results": (0.92, 0.88),
 "qualifying": (0.08, 0.34), "lap_times": (0.36, 0.40), "pit_stops": (0.08, 0.10),
 "driver_standings": (0.36, 0.26) , "constructor_standings": (0.92, 0.26),
 "constructor_results": (0.92, 0.08), "practice_sessions": (0.64, 0.88),
}
KIND = {t: "dim" for t in ["seasons","circuits","races","drivers","constructors","status"]}
PKTXT = {"seasons":"year","circuits":"circuitId","races":"raceId","drivers":"driverId","constructors":"constructorId",
         "status":"statusId","results":"resultId","sprint_results":"resultId","qualifying":"qualifyId",
         "lap_times":"raceId+driverId+lap","pit_stops":"raceId+driverId+stop","driver_standings":"driverStandingsId",
         "constructor_standings":"constructorStandingsId","constructor_results":"constructorResultsId",
         "practice_sessions":"raceId+driverId+session"}
FK_EDGES = [("races","seasons"),("races","circuits"),("results","races"),("results","drivers"),("results","constructors"),
  ("results","status"),("sprint_results","races"),("sprint_results","drivers"),("sprint_results","constructors"),
  ("qualifying","races"),("qualifying","drivers"),("qualifying","constructors"),("lap_times","races"),("lap_times","drivers"),
  ("pit_stops","races"),("pit_stops","drivers"),("driver_standings","races"),("driver_standings","drivers"),
  ("constructor_standings","races"),("constructor_standings","constructors"),("constructor_results","races"),
  ("constructor_results","constructors"),("practice_sessions","races"),("practice_sessions","drivers")]

fig, ax = plt.subplots(figsize=(16, 9)); ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.03, 1.0); ax.axis("off")
for a, b in FK_EDGES:
    (x1, y1), (x2, y2) = POS[a], POS[b]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#9aa5b1", lw=0.8, alpha=0.7), zorder=1)
for t, (x, y) in POS.items():
    dim = KIND.get(t) == "dim"
    ax.add_patch(FancyBboxPatch((x-0.075, y-0.035), 0.15, 0.07, boxstyle="round,pad=0.005",
                 fc="#dbeafe" if dim else "#fde7c8", ec="#1d4ed8" if dim else "#c2410c", lw=1.5, zorder=2))
    ax.text(x, y+0.008, t, ha="center", va="center", fontsize=10, fontweight="bold", zorder=3)
    ax.text(x, y-0.016, f"PK: {PKTXT[t]}", ha="center", va="center", fontsize=7, color="#444", zorder=3)
ax.set_title("F1 schema — أزرق = Dimension ، برتقالي = Fact/تفصيلي ، السهم = FK (من الابن إلى الأب)", fontsize=13)
plt.tight_layout(); plt.show()

## 3.3 `fact_results` — الجدول المحوري
أعمدة مشتقة (كلها بدون تضخيم عدد الصفوف — نفس عدد صفوف `results`):

- `is_win / is_podium / is_points_finish / has_fastest_lap / started_on_pole`
- `is_finish / is_dnf / is_dsq / is_dns` من `status_group` (عبر `status`)
- `positions_gained = grid − positionOrder` (**موجب = كسب مراكز**) للمُصنَّفين فقط ولمن انطلق من الشبكة العادية
- `laps_completed_pct = laps ÷ أكبر عدد لفات في السباق` (نسبة إكمال السباق)
- `quali_position / best_q_ms / reached_q3` من `qualifying` (join فريد على `raceId, driverId`)
- `age_at_race` (بالسنوات) من `drivers.dob`

In [ ]:
races = T["races"]
ERA_BINS   = [1949, 1969, 1989, 2013, 2021, 2100]
ERA_LABELS = ["1950-69 Pioneer", "1970-89 Classic", "1990-2013 V10/V8", "2014-21 Hybrid", "2022+ Ground effect"]
races = races.assign(era=pd.cut(races["year"], ERA_BINS, labels=ERA_LABELS).astype("string"))   # عدّل الحدود لو تحب

res = T["results"].copy()
n0 = len(res)

# --- ربط السنة/الجولة/التاريخ ---
res = res.merge(races[["raceId", "year", "round", "date"]], on="raceId", how="left")

# --- ربط حالة الإنهاء ---
res = res.merge(T["status"][["statusId", "status_group"]], on="statusId", how="left")
sg = res["status_group"]
res["is_finish"] = sg.isin(["Finished", "Finished (Lapped)"])
res["is_dsq"]    = sg.eq("Disqualified")
res["is_dns"]    = sg.eq("DNS / Not classified")
res["is_dnf"]    = ~(res["is_finish"] | res["is_dsq"] | res["is_dns"])     # انسحاب فعلي (عطل/حادث/...)

# --- فوز / منصة / نقاط ---
pos = res["position"]
res["is_win"]           = pos.eq(1).fillna(False).astype(bool)
res["is_podium"]        = (pos <= 3).fillna(False).astype(bool)
res["is_points_finish"] = res["points"] > 0
res["has_fastest_lap"]  = res["rank"].eq(1).fillna(False).astype(bool)
res["started_on_pole"]  = res["grid_clean"].eq(1).fillna(False).astype(bool)

# --- مكاسب المراكز: فقط للمُصنَّفين الذين لهم grid حقيقي ---
ok = res["is_classified"] & res["grid_clean"].notna()
res["positions_gained"] = (res["grid_clean"].astype("float64") - res["positionOrder"]).where(ok).astype("Int16")

# --- نسبة إكمال السباق ---
res["race_laps"] = res.groupby("raceId")["laps"].transform("max").astype("int16")
res["laps_completed_pct"] = (res["laps"] / res["race_laps"]).round(4).astype("float32")

# --- التأهل (join فريد على raceId+driverId) ---
q = T["qualifying"][["raceId", "driverId", "position", "best_q_ms", "reached_q3"]].rename(columns={"position": "quali_position"})
assert not q.duplicated(["raceId", "driverId"]).any()
res = res.merge(q, on=["raceId", "driverId"], how="left")
res["reached_q3"] = res["reached_q3"].fillna(False).astype(bool)
res["has_quali_data"] = res["quali_position"].notna()

# --- العمر وقت السباق ---
res = res.merge(T["drivers"][["driverId", "dob"]], on="driverId", how="left")
res["age_at_race"] = ((res["date"] - res["dob"]).dt.days / 365.25).round(1).astype("float32")

fact_results = res[[
    "resultId", "raceId", "driverId", "constructorId", "statusId", "year", "round",
    "grid", "grid_clean", "grid_is_pitlane", "quali_position", "has_quali_data", "best_q_ms", "reached_q3",
    "position", "positionOrder", "points", "laps", "race_laps", "laps_completed_pct", "milliseconds",
    "fastest_lap_ms", "rank", "fastestLapSpeed",
    "is_classified", "is_finish", "is_dnf", "is_dsq", "is_dns", "is_win", "is_podium", "is_points_finish",
    "has_fastest_lap", "started_on_pole", "is_shared_drive", "positions_gained", "age_at_race",
]].copy()
fact_results["status_group"] = res["status_group"].astype("category")

assert len(fact_results) == n0, "الـ joins غيّرت عدد الصفوف!"
assert fact_results["resultId"].is_unique
print(f"fact_results: {fact_results.shape}  (نفس عدد صفوف results ✅)")
display(fact_results.sample(5, random_state=1)[["year","driverId","grid_clean","positionOrder","positions_gained","status_group","is_win","is_podium","laps_completed_pct"]])

## 3.4 `fact_driver_race_perf` — إحصاءات اللفات والـ Pit Stops
**Grain = (raceId, driverId)**. نستبعد `is_outlier_lap` و`is_outlier_stop` من المتوسطات (لكن نسجّل عددها).

In [ ]:
lt = T["lap_times"]
clean_laps = lt[~lt["is_outlier_lap"]]
lap_stats = clean_laps.groupby(["raceId", "driverId"], observed=True)["milliseconds"].agg(
    best_lap_ms="min", avg_lap_ms="mean", median_lap_ms="median", lap_std_ms="std", n_clean_laps="count").reset_index()
lap_stats[["avg_lap_ms", "median_lap_ms", "lap_std_ms"]] = lap_stats[["avg_lap_ms", "median_lap_ms", "lap_std_ms"]].round(0)
lap_n = lt.groupby(["raceId", "driverId"], observed=True).agg(n_laps_recorded=("lap", "count"), n_outlier_laps=("is_outlier_lap", "sum")).reset_index()
lap1 = lt.loc[lt["lap"] == 1, ["raceId", "driverId", "position"]].rename(columns={"position": "lap1_position"})
lap_stats = lap_n.merge(lap_stats, on=["raceId", "driverId"], how="left").merge(lap1, on=["raceId", "driverId"], how="left")

ps = T["pit_stops"]
valid = ps[~ps["is_outlier_stop"]]
pit_stats = valid.groupby(["raceId", "driverId"], observed=True).agg(
    n_pit_stops=("stop", "count"), total_pit_s=("duration_s", "sum"), avg_pit_s=("duration_s", "mean"),
    first_stop_lap=("lap", "min")).reset_index()
pit_out = ps.groupby(["raceId", "driverId"], observed=True)["is_outlier_stop"].sum().rename("n_outlier_stops").reset_index()
pit_stats = pit_out.merge(pit_stats, on=["raceId", "driverId"], how="left")
pit_stats["n_pit_stops"] = pit_stats["n_pit_stops"].fillna(0).astype("int8")      # سائق لديه توقفات كلها شاذة = 0 صالحة

fact_perf = lap_stats.merge(pit_stats, on=["raceId", "driverId"], how="outer")
fact_perf["has_lap_data"] = fact_perf["n_laps_recorded"].notna()
fact_perf["has_pit_data"] = fact_perf["n_outlier_stops"].notna()
for c in ["n_laps_recorded", "n_outlier_laps", "n_outlier_stops"]:
    fact_perf[c] = fact_perf[c].fillna(0).astype("int16")
assert not fact_perf.duplicated(["raceId", "driverId"]).any()
print("fact_driver_race_perf:", fact_perf.shape, "| lap data:", int(fact_perf.has_lap_data.sum()), "| pit data:", int(fact_perf.has_pit_data.sum()))
display(fact_perf.dropna(subset=["avg_pit_s"]).head(3))

## 3.5 `fact_race_summary` — ملخص كل سباق
نسبة الإكمال، الفائز، صاحب الـ Pole، وهل فاز. ملاحظة: الفائز يُحدَّد بـ `positionOrder == 1` (لأن 3 سباقات في الخمسينات انتهت بفوز مشترك لسائقَين ويظهر لهما `position = 1`).

In [ ]:
fr = fact_results
starters = fr[~fr["is_dns"]]
g = starters.groupby("raceId")
summ = pd.DataFrame({
    "starters": g.size(),
    "finishers": g["is_finish"].sum(),
    "dnf_count": g["is_dnf"].sum(),
    "dsq_count": fr.groupby("raceId")["is_dsq"].sum(),
    "race_laps": fr.groupby("raceId")["race_laps"].first(),
    "best_fastest_lap_ms": fr.groupby("raceId")["fastest_lap_ms"].min(),
}).reset_index()
summ["finish_rate"] = (summ["finishers"] / summ["starters"]).round(4)
summ["dnf_rate"]    = (summ["dnf_count"] / summ["starters"]).round(4)

winner = fr[fr["positionOrder"] == 1].drop_duplicates("raceId")[["raceId", "driverId", "constructorId", "grid_clean"]]
winner.columns = ["raceId", "winner_driverId", "winner_constructorId", "winner_grid"]
pole = fr[fr["grid_clean"] == 1].drop_duplicates("raceId")[["raceId", "driverId"]].rename(columns={"driverId": "pole_driverId"})
summ = summ.merge(winner, on="raceId", how="left").merge(pole, on="raceId", how="left")
summ["pole_won"] = (summ["winner_driverId"] == summ["pole_driverId"]).where(summ["pole_driverId"].notna())

# تعداد الـ Pit Stops (فقط للسباقات التي لها بيانات pit)
pit_race = fact_perf[fact_perf["has_pit_data"]].groupby("raceId").agg(
    drivers_with_pit_data=("driverId", "nunique"), total_pit_stops=("n_pit_stops", "sum")).reset_index()
pit_race["avg_stops_per_driver"] = (pit_race["total_pit_stops"] / pit_race["drivers_with_pit_data"]).round(2)
summ = summ.merge(pit_race[["raceId", "total_pit_stops", "avg_stops_per_driver"]], on="raceId", how="left")

# تحويل pole_won لـ boolean nullable
summ["pole_won"] = summ["pole_won"].astype("boolean")
summ = summ.merge(races[["raceId", "year", "round", "circuitId", "era"]], on="raceId", how="left")
fact_race_summary = summ
assert fact_race_summary["raceId"].is_unique and len(fact_race_summary) == races.shape[0]
print("fact_race_summary:", fact_race_summary.shape)
print("نسبة فوز صاحب الـ Pole (كل التاريخ): {:.1%}".format(fact_race_summary["pole_won"].dropna().astype(float).mean()))

## 3.6 تجميعات الموسم: `agg_driver_season` و `agg_constructor_season`
- النقاط والمركز النهائي من **آخر جولة في الموسم** في جداول الـ standings (تشمل نقاط الـ Sprint، بخلاف مجموع `results.points`).
- `champion = (final_position == 1)`.

In [ ]:
last_race = races.loc[races.groupby("year")["round"].idxmax(), ["raceId", "year"]].rename(columns={"raceId": "last_raceId"})

def season_agg(fr, key):
    g = fr.groupby([key, "year"])
    out = g.agg(
        starts=("resultId", "count"), wins=("is_win", "sum"), podiums=("is_podium", "sum"),
        poles=("started_on_pole", "sum"), fastest_laps=("has_fastest_lap", "sum"),
        dnfs=("is_dnf", "sum"), points_from_races=("points", "sum"),
        finish_count=("is_finish", "sum"), avg_grid=("grid_clean", lambda s: s.astype("float64").mean()),
        avg_finish=("positionOrder", "mean")).reset_index()
    out["finish_rate"] = (out["finish_count"] / out["starts"]).round(4)
    out["win_rate"] = (out["wins"] / out["starts"]).round(4)
    out["podium_rate"] = (out["podiums"] / out["starts"]).round(4)
    return out.drop(columns="finish_count")

# ---- السائقون ----
ads = season_agg(fact_results, "driverId")
fin = T["driver_standings"].merge(last_race, left_on="raceId", right_on="last_raceId")[["driverId", "year", "points", "position", "wins"]]
fin = fin.rename(columns={"points": "final_points", "position": "final_position", "wins": "standings_wins"})
agg_driver_season = ads.merge(fin, on=["driverId", "year"], how="left")
agg_driver_season["champion"] = agg_driver_season["final_position"].eq(1).fillna(False).astype(bool)

# ---- الفرق ----
acs = season_agg(fact_results, "constructorId")
finc = T["constructor_standings"].merge(last_race, left_on="raceId", right_on="last_raceId")[["constructorId", "year", "points", "position", "wins"]]
finc = finc.rename(columns={"points": "final_points", "position": "final_position", "wins": "standings_wins"})
agg_constructor_season = acs.merge(finc, on=["constructorId", "year"], how="left")
agg_constructor_season["champion"] = agg_constructor_season["final_position"].eq(1).fillna(False).astype(bool)

# ---- فحوصات منطقية ----
assert agg_driver_season.duplicated(["driverId", "year"]).sum() == 0
champs = agg_driver_season[agg_driver_season.champion].groupby("year").size()
print("سنوات لها بطل واحد بالضبط:", int((champs == 1).sum()), "من", races.year.nunique(), "| سنوات بلا بطل/بأكثر من بطل:", sorted(set(races.year.unique()) - set(champs[champs == 1].index)))
print("أبطال آخر 5 مواسم:")
names = T["drivers"].set_index("driverId")["driver_name"]
display(agg_driver_season[agg_driver_season.champion].sort_values("year").tail(5).assign(driver=lambda d: d.driverId.map(names))[["year", "driver", "wins", "final_points"]])

# مقارنة نقاط السباقات بنقاط الـ standings (الفروق متوقعة: Sprint 2021+ ونظام أفضل-N نتائج قديماً)
diff = agg_driver_season.dropna(subset=["final_points"]).assign(d=lambda d: (d.final_points - d.points_from_races).abs() > 0.01)
print("نسبة صفوف السائق-موسم التي تختلف فيها نقاط الـ standings عن مجموع results:", f"{diff.d.mean():.1%}", "(أغلبها سنوات الـ Sprint ونظام الـ best-N)")

## 3.7 `agg_circuit` — الحلبات: الانسحاب والإثارة واستراتيجية الـ Pit

In [ ]:
fr = fact_results
cls = fr[fr["positions_gained"].notna()].merge(races[["raceId", "circuitId"]], on="raceId")
overtake = cls.groupby("circuitId").agg(avg_abs_positions_change=("positions_gained", lambda s: s.astype("float64").abs().mean())).reset_index()

rs = fact_race_summary
circ = rs.groupby("circuitId").agg(
    n_races=("raceId", "count"), first_year=("year", "min"), last_year=("year", "max"),
    total_starters=("starters", "sum"), total_dnf=("dnf_count", "sum"),
    avg_finish_rate=("finish_rate", "mean"), pole_win_rate=("pole_won", lambda s: s.dropna().astype(float).mean()),
    avg_stops_per_driver=("avg_stops_per_driver", "mean"), n_races_with_pit_data=("avg_stops_per_driver", "count"),
).reset_index()
circ["dnf_rate"] = (circ["total_dnf"] / circ["total_starters"]).round(4)
agg_circuit = (circ.merge(overtake, on="circuitId", how="left")
                   .merge(T["circuits"][["circuitId", "name", "country_std"]].rename(columns={"name": "circuit_name"}), on="circuitId", how="left"))
print("agg_circuit:", agg_circuit.shape)
display(agg_circuit[agg_circuit.n_races >= 20].sort_values("dnf_rate", ascending=False).head(8)[
    ["circuit_name", "country_std", "n_races", "dnf_rate", "pole_win_rate", "avg_abs_positions_change", "avg_stops_per_driver"]])

## 3.8 فحوصات سلامة نهائية + حفظ المخرجات

In [ ]:
# --- sanity: أكثر الفائزين (يجب أن تبدو منطقية) ---
dn = T["drivers"].set_index("driverId")["driver_name"]; cn = T["constructors"].set_index("constructorId")["constructor_name"]
top_d = fact_results.groupby("driverId")[["is_win", "is_podium"]].sum().sort_values("is_win", ascending=False).head(5)
top_c = fact_results.groupby("constructorId")[["is_win", "is_podium"]].sum().sort_values("is_win", ascending=False).head(5)
display(top_d.rename(index=dn)); display(top_c.rename(index=cn))

# --- اتساق: مجموع الانتصارات ---
assert agg_driver_season["wins"].sum() == fact_results["is_win"].sum()
assert fact_race_summary["winner_driverId"].notna().all()
assert fact_results["laps_completed_pct"].between(0, 1.0001).all()
assert fact_results["positions_gained"].dropna().abs().max() < 40

# --- حفظ ---
OUT = dict(fact_results=fact_results, fact_driver_race_perf=fact_perf, fact_race_summary=fact_race_summary,
           agg_driver_season=agg_driver_season, agg_constructor_season=agg_constructor_season, agg_circuit=agg_circuit)
rows = []
for n, df in OUT.items():
    df = df.reset_index(drop=True)
    try: p = MODEL_DIR / f"{n}.parquet"; df.to_parquet(p, index=False)
    except ImportError: p = MODEL_DIR / f"{n}.csv"; df.to_csv(p, index=False)
    rows.append(dict(table=n, rows=len(df), cols=df.shape[1], size_MB=round(p.stat().st_size / 1024**2, 2)))
display(pd.DataFrame(rows).set_index("table"))
print("✅ المخرجات في:", MODEL_DIR)

## ⚠️ ملاحظات مهمة قبل المرحلة 4 / Power BI
1. **لا تجمع نقاط `fact_results` عبر الـ Standings**: للنقاط الرسمية للموسم استخدم `agg_*_season.final_points`.
2. `fact_driver_race_perf` له Grain مختلف عن `fact_results` (`raceId+driverId` بدل `resultId`) — اربطهما في Power BI عبر **Dim_Races + Dim_Drivers** وليس مباشرة (لتجنب Many-to-Many في 172 صف مشترك).
3. `positions_gained`, `avg_pit_s`, `best_q_ms` فيها **قيم فارغة مقصودة** (لا تغطية قبل 1994/1996/2011) — لا تعاملها كأصفار.
4. الفوز المشترك (3 سباقات في الخمسينات) يُحسب لكل من السائقين في `is_win`، بينما `fact_race_summary` يعطي فائزاً واحداً لكل سباق.